# TrustQueryNet Colab Runner

Use this notebook in browser Colab UI for Drive-backed runs and artifact persistence.

If you connect from VS Code to a Colab kernel, set `USE_DRIVE = False` in the config cell and run against `/content` instead.

In [ ]:
REPO_URL = 'https://github.com/stelioszach03/TrustQueryNet.git'
USE_DRIVE = True
DRIVE_MOUNT_POINT = '/content/drive'
HAM_DIR = '/content/drive/MyDrive/HAM10000' if USE_DRIVE else '/content/HAM10000'
RESULTS_ROOT = '/content/drive/MyDrive/TrustQueryNet/artifacts' if USE_DRIVE else '/content/TrustQueryNet/artifacts'
PROJECT_ROOT = '/content/TrustQueryNet'
DOWNLOAD_HAM10000 = True

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT_POINT)
    print(f'Drive mounted at {DRIVE_MOUNT_POINT}')
else:
    print('Skipping Drive mount; using ephemeral /content paths.')

In [ ]:
import os
os.chdir('/content')
!rm -rf {PROJECT_ROOT}
!git clone {REPO_URL} {PROJECT_ROOT}
%cd {PROJECT_ROOT}
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps

In [ ]:
if DOWNLOAD_HAM10000:
    !python scripts/download_ham10000.py --output-root {HAM_DIR}
else:
    print('Skipping HAM10000 download. Expecting existing files under HAM_DIR.')

In [ ]:
from pathlib import Path

ham_dir = Path(HAM_DIR)
metadata_csv = ham_dir / 'HAM10000_metadata.csv'
image_dir = ham_dir / 'images'
results_root = Path(RESULTS_ROOT)
results_root.mkdir(parents=True, exist_ok=True)

if not metadata_csv.exists():
    raise FileNotFoundError(f'Missing metadata CSV: {metadata_csv}')
if not image_dir.exists():
    raise FileNotFoundError(f'Missing images directory: {image_dir}')

jpg_count = len(list(image_dir.glob('*.jpg')))
print(f'Metadata CSV: {metadata_csv}')
print(f'Images dir:   {image_dir}')
print(f'JPG count:    {jpg_count}')
print(f'Results root: {results_root}')

In [ ]:
from pathlib import Path
import yaml

project_root = Path(PROJECT_ROOT)
ham_dir = Path(HAM_DIR)
results_root = Path(RESULTS_ROOT)

def build_colab_config(base_name: str, *, num_workers: int = 2):
    base_path = project_root / 'configs' / base_name
    cfg = yaml.safe_load(base_path.read_text())
    cfg['dataset']['metadata_csv'] = str(ham_dir / 'HAM10000_metadata.csv')
    cfg['dataset']['image_dir'] = str(ham_dir / 'images')
    cfg['dataset']['split_csv'] = str(ham_dir / 'splits.csv')
    cfg['dataset']['save_split_csv'] = str(ham_dir / 'splits.csv')
    output_subdir = Path(str(cfg['output_dir'])).name
    cfg['output_dir'] = str(results_root / 'runs' / output_subdir)
    cfg['num_workers'] = num_workers
    out_name = f'colab_{base_name}'
    out_path = project_root / 'configs' / out_name
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(base_name, {
        'experiment_name': cfg['experiment_name'],
        'output_dir': cfg['output_dir'],
        'epochs': cfg['training']['epochs'],
        'loss': cfg['training']['loss'],
        'label_smoothing': cfg['training'].get('label_smoothing', 0.0),
        'sampler': cfg['training'].get('sampler', 'shuffle'),
        'active_learning': cfg['active_learning'],
    })
    return out_path

pilot_cfg_path = build_colab_config('pilot_ham10000.yaml', num_workers=2)
full_cfg_path = build_colab_config('full_ham10000_convnext.yaml', num_workers=2)

print(f'Pilot config: {pilot_cfg_path}')
print(f'Full config:  {full_cfg_path}')

In [ ]:
!python scripts/prepare_ham10000.py --metadata-csv {HAM_DIR}/HAM10000_metadata.csv --image-dir {HAM_DIR}/images --split-csv {HAM_DIR}/splits.csv --report-json {RESULTS_ROOT}/ham10000_dataset_report.json

In [ ]:
# Pilot run first if you want a quick sanity check:
# !python scripts/run_experiment.py --config configs/colab_pilot_ham10000.yaml

In [ ]:
# Current best full run:
# !python scripts/run_experiment.py --config configs/colab_full_ham10000_convnext.yaml

In [ ]:
# Export a final paper bundle after the runs finish:
# !python scripts/export_results_bundle.py \
#   --runs-root {RESULTS_ROOT}/runs \
#   --run pilot-ham10000 \
#   --run full-ham10000-convnext-balanced \
#   --output-root /content/drive/MyDrive/TrustQueryNet/exports \
#   --bundle-name trustquerynet-paper-bundle